# Kaggle: train the pack model (smoke / short runs only)

The 0.942 recipe is 400 epochs at ~20 min/epoch on an RTX 5090 -- far beyond a Kaggle session. This notebook exists to run the *same* training code for a few epochs on Kaggle hardware (smoke test, or a short fine-tune with `--max-hours`). Real training runs on the BC cluster (`scripts/slurm_train.sbatch`). Attach the competition data, `cell-tracking-src`, and `biohub-tracking-wheels`; GPU on; internet may be on.

In [ ]:
import time, os, subprocess, sys
from pathlib import Path
SESSION_STARTED = time.time()
SRC_DATASET = Path('/kaggle/input/cell-tracking-src')
WHEELS_DATASET = Path('/kaggle/input/biohub-tracking-wheels')
assert SRC_DATASET.exists(), f'code Dataset not attached at {SRC_DATASET}'
sys.path.insert(0, str(SRC_DATASET / 'src'))
SRC_ENV = {**os.environ, 'PYTHONPATH': str(SRC_DATASET / 'src'), 'POLARS_PREFER_PKG': '32'}
find_links = []
if WHEELS_DATASET.exists():
    for d in sorted({p.parent for p in WHEELS_DATASET.rglob('*.whl')}):
        find_links += ['--find-links', str(d)]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index' if find_links else '--quiet', *find_links,
                'tracksdata', 'ilpy', 'pyscipopt', 'polars', 'polars_runtime_32', 'rustworkx', 'sqlalchemy',
                'dask', 'imagecodecs', 'pyarrow', 'blosc2', 'zarr'], check=True)


In [ ]:
# 1. Decimated-frame cache (once per session; ~10 GB for all 199 volumes, so limit for smoke runs).
subprocess.run([sys.executable, str(SRC_DATASET / 'scripts' / 'build_cache.py'),
                '--cache-dir', '/kaggle/working/cache_pack', '--limit', '40'], check=True, env=SRC_ENV)


In [ ]:
# 2. Train with the pack recipe (AdamW 1e-4, batch 8, no scheduler), held-out split, val_score selection.
#    --max-hours keeps the session inside Kaggle's limit; resume the same --out to continue in a new session.
subprocess.run([sys.executable, str(SRC_DATASET / 'scripts' / 'train.py'),
                '--cache-dir', '/kaggle/working/cache_pack', '--limit-volumes', '40',
                '--epochs', '400', '--max-hours', '8.5', '--eval-tracking-every', '5', '--save-every', '25',
                '--seed', '0', '--out', '/kaggle/working/pack_s0.pt'], check=True, env=SRC_ENV)
